## 1. Instalación de dependencias

### Opcion 1: Si tenes una sola instalacion de Python

In [2]:
# pip install requests beautifulsoup4 pandas

### Opcion 2: Si tenes varias instalaciones de Python

In [1]:
# # Ejecutar esta celda una sola vez para instalar los paquetes
# # Despues de instalarse, se reinicia el kernel para que esten disponibles en las siguientes celdas

# import sys, subprocess, importlib

# PACKAGES = ["beautifulsoup4", "pandas"]

# def install_and_restart(packages):
#     installed_any = False
#     for pkg in packages:
#         mod = pkg.split("[")[0].replace("-", "_")  
#         try:
#             importlib.import_module(mod)
#         except ImportError:
#             print(f"Installing {pkg}...")
#             subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
#             installed_any = True
    
#     if installed_any:
#         print("\n✅ Packages installed. Restarting kernel...")
#         import IPython
#         IPython.Application.instance().kernel.do_shutdown(restart=True)
#     else:
#         print("✅ All packages already installed, no restart needed.")

# install_and_restart(PACKAGES)

## 2. Imports y funciones auxiliares

In [2]:
# ============================================================
# ARGENPROP SCRAPPER - Departamentos en Venta Capital Federal
# ============================================================

import os
import re
import time
import requests
import pandas as pd
from datetime import date
from bs4 import BeautifulSoup

# ──────────────────────────────────────────────
# UTILIDADES
# ──────────────────────────────────────────────

def clean_text(text):
    if not text: return None
    text = text.replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip() or None


def extract_smart_features(row):
    texto = (str(row.get('Descripción', '')) + " " + str(row.get('Detalles', ''))).lower()
    return pd.Series({
        "Amenities":         sum(1 for x in ["amenities", "piscina", "pileta", "sum", "parrilla", "gym", "sauna", "laundry"] if x in texto),
        "Losa_Central":      1 if any(x in texto for x in ["losa radiante", "calefacción central", "caldera central", "piso radiante"]) else 0,
        "Aire_Acond":        1 if any(x in texto for x in ["aire acondicionado", "split", " a/c", "frío-calor"]) else 0,
        "Apto_Credito":      1 if "apto crédito" in texto or "apto credito" in texto else 0,
        "Cochera":           1 if any(x in texto for x in ["cochera", "espacio guarda coche", "estacionamiento", "guarda coche"]) else 0,
        "Seguridad":         1 if any(x in texto for x in ["vigilancia", "seguridad 24", "tótem", "totem", "encargado", "portero", "guardia", "cámara", "monitoreo"]) else 0,
        "Luminoso":          1 if any(x in texto for x in ["luminoso", "todo luz", "vista abierta", "vista panorámica", "muy soleado", "soleado"]) else 0,
        "Balcon_Aterrazado": 1 if "aterrazado" in texto or "balcón terraza" in texto else 0,
    })


MESES = r'(enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|octubre|noviembre|diciembre)'


def parse_address(address_raw):
    calle = altura = piso = None
    if not address_raw:
        return calle, altura, piso
    try:
        # En Argenprop el piso viene como "ARAOZ 1200, Piso 8" — separarlo antes de parsear
        piso_match = re.search(r',\s*[Pp]iso\s+(\w+)', address_raw)
        if piso_match:
            piso = piso_match.group(1).strip().replace('º', '').replace('°', '')
            address_raw = address_raw[:piso_match.start()].strip()  # quitar ", Piso 8" del string

        cleaned = re.sub(r'\bal\s+', '', address_raw, flags=re.IGNORECASE)

        # 2. Si tiene guión, quedarse con fragmento que tenga altura válida (>= 100)
        if '-' in cleaned:
            fragmentos = [f.strip() for f in cleaned.split('-')]
            candidato = None
            for frag in fragmentos:
                if re.search(r'\b\d{3,5}\b', frag):
                    candidato = frag
                    break
            if candidato:
                cleaned = candidato
            else:
                return None, None, piso  # ← conservar piso si ya lo extrajimos

        # 3. Intentar capturar piso inline: "Junín 1615 piso 13" o "Junín 1615 PB"
        if piso is None:
            match_piso = re.search(
                r'([A-Za-záéíóúÁÉÍÓÚñÑü\s\.]+?)\s+(\d{2,5})\s+(piso\s+\d+|PB|pb|p\.b\.)',
                cleaned, re.IGNORECASE
            )
            if match_piso:
                num = int(match_piso.group(2))
                if not (1800 <= num <= 1950 and re.search(MESES, cleaned[:match_piso.start(2)], re.IGNORECASE)):
                    calle    = match_piso.group(1).strip().strip('-').strip('.').title()
                    altura   = match_piso.group(2).strip()
                    piso_raw = match_piso.group(3).strip()
                    piso     = re.search(r'\d+', piso_raw).group(0).replace('º', '').replace('°', '') if re.search(r'\d+', piso_raw) else "PB"
                    return calle, altura, piso

        # 4. Iterar números y encontrar altura real
        for m in re.finditer(r'(\d{2,5})(?:[.,\s\-]|$)', cleaned + ' '):
            num = int(m.group(1))
            if num < 100:
                continue
            contexto_previo = cleaned[:m.start()].lower()
            es_año_historico = (1800 <= num <= 1950) and bool(re.search(MESES, contexto_previo, re.IGNORECASE))
            if es_año_historico:
                continue
            calle_raw = cleaned[:m.start()].strip().strip('-').strip('.').strip()
            if not calle_raw:
                continue
            calle  = calle_raw.title()
            altura = str(num)
            return calle, altura, piso

    except:
        pass
    return None, None, piso


def parse_card_argenprop(item):
    # Link
    link_tag = item.find('a', class_='card')
    if not link_tag:
        return None
    link = "https://www.argenprop.com" + link_tag['href']

    # Precio y expensas
    price_block = item.find('p', class_='card__price')
    price_text  = clean_text(price_block.text) if price_block else ""
    p_match  = re.search(r'(USD|ARS|\$)\s?([\d.]+)', price_text or "")
    precio   = p_match.group(0) if p_match else "Consultar"
    e_match  = re.search(r'\+\s?\$?\s?([\d.]+)', price_text or "")
    expensas = e_match.group(0) if e_match else None
    expensas = expensas[2:] if expensas else None

    # Dirección — viene como "ARAOZ 1200, Piso 8"
    addr_tag    = item.find('p', class_='card__address')
    raw_address = clean_text(addr_tag.text) if addr_tag else None
    calle, altura, piso = parse_address(raw_address)

    # Barrio — en Argenprop está en card__title--primary: "Depto en Venta en Palermo, Capital Federal"
    barrio_tag = item.find('p', class_='card__title--primary')
    barrio = None
    if barrio_tag:
        barrio_text = clean_text(barrio_tag.text)
        # Extraer solo el barrio: "Departamento en Venta en Palermo, Capital Federal" → "Palermo"
        b_match = re.search(r'\ben\s+([^,]+),', barrio_text, re.IGNORECASE)
        barrio = (b_match.group(1).strip()[21:] if 'Temporal' in b_match.group(1).strip() else (b_match.group(1).strip()[12:] if 'Alquiler' in b_match.group(1).strip() else b_match.group(1).strip()[9:]))if b_match else barrio_text
    
    # Características
    feat_tag = item.find('ul', class_='card__main-features')
    features = clean_text(feat_tag.get_text(separator=' ')) if feat_tag else None

    # Descripción corta
    desc_tag = item.find('p', class_='card__description') or item.find('p', class_='card__info')
    desc     = clean_text(desc_tag.text) if desc_tag else None

    return link, precio, expensas, calle, altura, piso, barrio, features, desc

print("✅ Funciones cargadas correctamente.")


✅ Funciones cargadas correctamente.


## 3. Función principal del scrapper

In [10]:
def run_scrapper_argenprop(enlace, operacion, max_pages=3, start_page=1, initial_data=None, initial_seen=None):
    """
    Scrapper de Argenprop con detección de CAPTCHA y capacidad de reanudación.

    Parámetros extra respecto a la versión anterior:
      start_page    : página desde la que arrancar (útil para reanudar tras CAPTCHA)
      initial_data  : lista de registros ya scrapeados en una ejecución anterior
      initial_seen  : set de links ya vistos en una ejecución anterior

    Cuando se detecta un CAPTCHA, la función:
      1. Guarda el progreso parcial en un TSV
      2. Imprime instrucciones claras
      3. Retorna un dict con todo lo necesario para reanudar

    Para reanudar después de resolver el CAPTCHA, llamar así:
      resultado = run_scrapper_argenprop("https://...", "venta", max_pages=200)
      # → CAPTCHA detectado en página 100, se guardó progreso parcial
      # → Resolvé el CAPTCHA en el navegador y presioná Enter aquí para continuar...
      # (después de resolver y presionar Enter, continúa solo)
    """
    BASE_URL   = enlace
    HEADERS    = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"}
    OUTPUT_DIR = "../../data/raw"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Retomar estado previo si se pasó como argumento
    all_data   = list(initial_data) if initial_data else []
    seen_links = set(initial_seen)  if initial_seen  else set()

    for i in range(start_page, max_pages + 1):
        url = f"{BASE_URL}?pagina-{i}" if i > 1 else BASE_URL
        print(f"\n--- PÁGINA {i} ---")
        print(f"URL: {url}")

        try:
            r    = requests.get(url, headers=HEADERS, timeout=15)
            soup = BeautifulSoup(r.content, 'html.parser')
            items = soup.find_all('div', class_='listing__item')

            # ── Detección de CAPTCHA ───────────────────────────────────────
            # Argenprop muestra CAPTCHA cuando no hay items pero la página
            # sí devolvió HTML. Señales adicionales: presencia de palabras
            # clave de verificación en el texto de la página.
            if not items:
                page_text = soup.get_text().lower()
                es_captcha = any(x in page_text for x in [
                    "verificar", "verify", "captcha", "robot", "humano",
                    "human", "cloudflare", "challenge", "security check"
                ])

                if es_captcha:
                    # 1. Guardar progreso parcial
                    partial_filename = f"argenprop_{operacion}_PARCIAL_{int(time.time())}.tsv"
                    partial_filepath = os.path.join(OUTPUT_DIR, partial_filename)

                    if all_data:
                        df_partial = pd.DataFrame(all_data)
                        df_partial = df_partial.replace("N/A", None)
                        features_df = df_partial.apply(extract_smart_features, axis=1)
                        df_partial = pd.concat([df_partial, features_df], axis=1)
                        df_partial.to_csv(partial_filepath, sep='\t', index=False, encoding='utf-8-sig')
                        print(f"\n💾 Progreso parcial guardado: {partial_filepath}")
                        print(f"   ({len(all_data)} propiedades scrapeadas hasta ahora)")
                    
                    # 2. Instrucciones claras
                    print(f"\n{'='*60}")
                    print(f"🔒 CAPTCHA DETECTADO en página {i}")
                    print(f"{'='*60}")
                    print(f"""
   PASO 1 — Abrí este link en tu navegador y resolvé el CAPTCHA:
            {url}

   PASO 2 — Una vez que veas las propiedades cargadas, copiá
            las cookies del navegador:
            → F12 → pestaña "Application" (Chrome) o "Storage" (Firefox)
            → Cookies → https://www.argenprop.com
            → Buscá la cookie llamada "cf_clearance" (u otra con "cf_")
            → Copiá su valor (es un string largo)

   PASO 3 — Pegalo acá cuando se te pida (o dejalo vacío si no
            encontrás ninguna cookie relevante y probamos igual)\n""")

                    # 3. Pedir cookie al usuario
                    cf_clearance = input("   Pegá el valor de cf_clearance (o Enter para saltear): ").strip()

                    # Construir cookies dict para el reintento
                    cookies_reintento = {}
                    if cf_clearance:
                        cookies_reintento["cf_clearance"] = cf_clearance
                        print("   ✅ Cookie recibida, reintentando con ella...")
                    else:
                        print("   ⚠️  Sin cookie — reintentando igual (puede fallar)...")

                    # 4. Reintentar con las cookies
                    print(f"\n🔄 Reintentando página {i}...")
                    time.sleep(5)

                    r2    = requests.get(url, headers=HEADERS, cookies=cookies_reintento, timeout=15)
                    soup2 = BeautifulSoup(r2.content, 'html.parser')
                    items = soup2.find_all('div', class_='listing__item')

                    if not items:
                        # Último intento: pedir el string completo de cookies del navegador
                        print("\n   ❌ Seguimos sin resultados.")
                        print("   Intentá copiar TODAS las cookies como string:")
                        print("   → F12 → pestaña 'Network' → recargá la página")
                        print("   → Click en el primer request a argenprop.com")
                        print("   → 'Request Headers' → copiá el valor completo del header 'Cookie'")
                        cookie_str = input("\n   Pegá el header Cookie completo (o Enter para abandonar): ").strip()

                        if cookie_str:
                            # Parsear "nombre=valor; nombre2=valor2" → dict
                            cookies_reintento = {}
                            for part in cookie_str.split(';'):
                                if '=' in part:
                                    k, v = part.strip().split('=', 1)
                                    cookies_reintento[k.strip()] = v.strip()

                            r3    = requests.get(url, headers=HEADERS, cookies=cookies_reintento, timeout=15)
                            soup3 = BeautifulSoup(r3.content, 'html.parser')
                            items = soup3.find_all('div', class_='listing__item')
                            soup2 = soup3

                    if not items:
                        print("\n⛔ No se pudo recuperar la sesión tras el CAPTCHA.")
                        print(f"   Corré el scrapper desde la página {i} una vez que se haya enfriado la IP:")
                        print(f"   run_scrapper_argenprop(..., start_page={i}, max_pages={max_pages})")
                        break

                    # Reemplazar soup con el nuevo para que el loop de cards funcione
                    soup = soup2
                    # Guardar cookies para usarlas en el resto de las páginas
                    HEADERS["Cookie"] = "; ".join(f"{k}={v}" for k, v in cookies_reintento.items())
                    print(f"✅ Sesión recuperada. Continuando desde página {i}...")

                else:
                    # No hay items y no es CAPTCHA → fin real del listado
                    print("⚠️ No se encontraron items. Fin del listado.")
                    break
            # ── Fin detección CAPTCHA ──────────────────────────────────────

            print(f"  → {len(items)} propiedades encontradas")

            for item in items:
                try:
                    result = parse_card_argenprop(item)
                    if not result:
                        continue
                    link, precio, expensas, calle, altura, piso, barrio, features, desc = result

                    if link in seen_links:
                        continue
                    seen_links.add(link)

                    print(f"  → {calle} {altura} | {precio}")

                    all_data.append({
                        "Fecha_Scraping": date.today().isoformat(),
                        "Posting_ID":     item.get('id'),
                        "Sito":           'argenprop',
                        "Operación":      operacion,
                        "Precio":         precio,
                        "Expensas":       expensas,
                        "Calle":          calle,
                        "Altura":         altura,
                        "Piso":           piso,
                        "Barrio":         barrio,
                        "Detalles":       features,
                        "Descripción":    desc,
                        "Link":           link,
                    })
                except Exception as e:
                    print(f"    ⚠️ Error en card: {e}")
                    continue

            time.sleep(1.5)

        except Exception as e:
            print(f"❌ Error en página {i}: {e}")
            break

    # ── Guardado final ─────────────────────────────────────────────────────
    if not all_data:
        print("⚠️ No se obtuvieron datos.")
        return None

    df = pd.DataFrame(all_data)
    df = df.replace("N/A", None)
    features_df = df.apply(extract_smart_features, axis=1)
    df = pd.concat([df, features_df], axis=1)

    filename = f"argenprop_{operacion}_{int(time.time())}.tsv"
    filepath = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(filepath, sep='\t', index=False, encoding='utf-8-sig')
    print(f"\n✅ ¡Éxito! Guardado en: {filepath}")
    return df

print("✅ run_scrapper_argenprop() lista.")


✅ run_scrapper_argenprop() lista.


## 4. ▶️ Ejecutar el scrapper

# Alquiler

In [11]:
df_alquiler = run_scrapper_argenprop("https://www.argenprop.com/departamentos/alquiler/capital-federal", 'alquiler', max_pages=3)


--- PÁGINA 1 ---
URL: https://www.argenprop.com/departamentos/alquiler/capital-federal
  → 20 propiedades encontradas
  → Gurruchaga 1100 | $ 490.000
  → Niceto Vega 5900 | $ 890.000
  → El Salvador 5500 | $ 1.600.000
  → Honduras 4600 | $ 900.000
  → Agüero 1000 | $ 590.000
  → Fitz Roy 1400 | $ 550.000
  → Rodriguez Peña 2000 | USD 1.400
  → Montevideo 1100 | $ 550.000
  → Juncal 2900 | $ 800.000
  → San Luis 3400 | $ 500.000
  → Gallo 900 | $ 550.000
  → La Pampa 1700 | USD 3.200
  → Ortiz De Ocampo 2500 | $ 2.200.000
  → Guise 1600 | $ 860.000
  → Gascón 1200 | $ 690.000
  → Avenida General Las Heras 4000 | $ 950.000
  → Uriarte 1400 | $ 650.000
  → República De Eslovenia 1900 | $ 700.000
  → Soldado De La Independencia 700 | $ 950.000
  → Humboldt 2400 | USD 1.650

--- PÁGINA 2 ---
URL: https://www.argenprop.com/departamentos/alquiler/capital-federal?pagina-2
  → 20 propiedades encontradas
  → Yerbal 2700 | $ 1.100.000
  → Rivadavia 7113 | $ 800.000
  → Doctor Tomás Manuel De Anc

In [5]:
if df_alquiler is not None:
    print(f"Total propiedades: {len(df_alquiler)}")
    display(df_alquiler.head(10))

    cols = ["Losa_Central","Aire_Acond","Apto_Credito",
            "Cochera","Seguridad","Luminoso","Balcon_Aterrazado"]
    print("\n📊 % con cada característica:")
    print((df_alquiler[cols].mean() * 100).round(1).astype(str) + "%")

Total propiedades: 60


,Fecha_Scraping,Posting_ID,Sito,Operación,Precio,Expensas,Calle,Altura,Piso,Barrio,...,Descripción,Link,Amenities,Losa_Central,Aire_Acond,Apto_Credito,Cochera,Seguridad,Luminoso,Balcon_Aterrazado
0,2026-04-14,18114368,argenprop,alquiler,$ 490.000,$280.000,Gurruchaga,1100,2,Palermo Viejo,...,Descubra la excelencia de vivir e invertir en ...,https://www.argenprop.com/departamento-en-alqu...,3,0,0,0,0,1,0,1
1,2026-04-14,19120155,argenprop,alquiler,$ 890.000,None,Niceto Vega,5900,3,Palermo Soho,...,Alquiler - monoambiente luminoso en palermo ub...,https://www.argenprop.com/departamento-en-alqu...,5,0,1,0,0,0,1,0
2,2026-04-14,10779484,argenprop,alquiler,$ 1.600.000,$365.000,El Salvador,5500,3,Palermo Hollywood,...,Alquiler: $ 1.600.000 ó u$s 1.150. Expensas: $...,https://www.argenprop.com/departamento-en-alqu...,2,0,0,0,1,0,1,0
3,2026-04-14,19226675,argenprop,alquiler,$ 900.000,None,Honduras,4600,6,Palermo,...,1 ambiente con terraza única en palermo - amoe...,https://www.argenprop.com/departamento-en-alqu...,5,0,0,0,0,0,1,0
4,2026-04-14,19432129,argenprop,alquiler,$ 590.000,$165.000,Agüero,1000,None,Recoleta,...,Dueño excelente departamento de 2 ambientes en...,https://www.argenprop.com/departamento-en-alqu...,0,0,0,0,0,0,1,0
5,2026-04-14,18816139,argenprop,alquiler,$ 550.000,$210.000,Fitz Roy,1400,None,Palermo Hollywood,...,El departamento consta de: * ambiente unico co...,https://www.argenprop.com/departamento-en-alqu...,0,0,1,0,0,0,0,1
6,2026-04-14,17789860,argenprop,alquiler,USD 1.400,$269.000,Rodriguez Peña,2000,2,Recoleta,...,•Espectacular departamento de categoria de 3 a...,https://www.argenprop.com/departamento-en-alqu...,0,0,0,0,0,0,1,0
7,2026-04-14,19202770,argenprop,alquiler,$ 550.000,$187.000,Montevideo,1100,None,Recoleta,...,"Dueño alquila, montevideo al 1100, entre avda....",https://www.argenprop.com/departamento-en-alqu...,0,0,1,0,0,0,1,0
8,2026-04-14,19409129,argenprop,alquiler,$ 800.000,$230.000,Juncal,2900,4,Recoleta,...,Dueño alquila depto 2 ambientes en recoleta. D...,https://www.argenprop.com/departamento-en-alqu...,0,0,1,0,0,0,1,0
9,2026-04-14,19260608,argenprop,alquiler,$ 500.000,$110.000,San Luis,3400,None,Palermo,...,"Dos ambientes muy luminosos, lateral sin balco...",https://www.argenprop.com/departamento-en-alqu...,0,0,0,0,0,0,1,0



📊 % con cada característica:
Losa_Central         18.3%
Aire_Acond           38.3%
Apto_Credito          0.0%
Cochera              15.0%
Seguridad            11.7%
Luminoso             55.0%
Balcon_Aterrazado     5.0%
dtype: object


# Alquiler Temporal

In [6]:
df_alquiler_temporal = run_scrapper_argenprop('https://www.argenprop.com/departamentos/alquiler-temporal/capital-federal', 'alquiler_temporal', max_pages=3)


--- PÁGINA 1 ---
URL: https://www.argenprop.com/departamentos/alquiler-temporal/capital-federal
  → 20 propiedades encontradas
  → Güemes 4400 | USD 550
  → Manuel Nicolas Savio 400 | USD 950
  → Av Gral Luis Maria Campos 1100 | $ 1.300.000
  → Concepción Arenal 3500 | $ 650.000
  → Coronel Diaz 1500 | USD 1.100
  → Av. Coronel Díaz 2400 | USD 1.250
  → Av. Coronel Díaz 2400 | USD 1.250
  → Ayacucho 2100 | USD 1.400
  → José Andrés Pacheco De Melo 1800 | USD 650
  → Guatemala 5653 | USD 1.500
  → Presidente José Evaristo Uriburu 1600 | Consultar
  → Delfin Huergo 300 | USD 1.150
  → Concepción Arenal 2300 | USD 1.000
  → Ecuador 1500 | USD 1.100
  → Pacheco De Melo, Jose 2700 | USD 550
  → None None | USD 680
  → Quirno Costa 1200 | $ 550
  → Rodriguez Peña 1400 | USD 1.500
  → Av Gral Luis Maria Campos 1100 | $ 1.300.000
  → Fitz Roy 2300 | USD 1.800

--- PÁGINA 2 ---
URL: https://www.argenprop.com/departamentos/alquiler-temporal/capital-federal?pagina-2
  → 20 propiedades encontrada

In [7]:
if df_alquiler_temporal is not None:
    print(f"Total propiedades: {len(df_alquiler_temporal)}")
    display(df_alquiler_temporal.head(10))

    cols = ["Losa_Central","Aire_Acond","Apto_Credito",
            "Cochera","Seguridad","Luminoso","Balcon_Aterrazado"]
    print("\n📊 % con cada característica:")
    print((df_alquiler_temporal[cols].mean() * 100).round(1).astype(str) + "%")

Total propiedades: 60


,Fecha_Scraping,Posting_ID,Sito,Operación,Precio,Expensas,Calle,Altura,Piso,Barrio,...,Descripción,Link,Amenities,Losa_Central,Aire_Acond,Apto_Credito,Cochera,Seguridad,Luminoso,Balcon_Aterrazado
0,2026-04-14,15506054,argenprop,alquiler_temporal,USD 550,$295.000,Güemes,4400,None,Palermo Soho,...,Green tower! Impecable monoambiente! Balcón al...,https://www.argenprop.com/departamento-en-alqu...,4,0,1,0,0,1,1,0
1,2026-04-14,18402372,argenprop,alquiler_temporal,USD 950,None,Manuel Nicolas Savio,400,4,Las Cañitas,...,Excelente unidad. Amplio y luminoso departamen...,https://www.argenprop.com/departamento-en-alqu...,0,0,1,0,0,0,1,0
2,2026-04-14,11420984,argenprop,alquiler_temporal,$ 1.300.000,$325.000,Av Gral Luis Maria Campos,1100,2,Las Cañitas,...,Alquiler: $ 1.300.000 ó u$s 950. Expensas: $ 3...,https://www.argenprop.com/departamento-en-alqu...,0,0,1,0,0,1,0,0
3,2026-04-14,19368839,argenprop,alquiler_temporal,$ 650.000,$199.600,Concepción Arenal,3500,None,Palermo Hollywood,...,Alquiler monoambiente amoblado en palermo con ...,https://www.argenprop.com/departamento-en-alqu...,5,1,1,0,0,0,0,0
4,2026-04-14,18676695,argenprop,alquiler_temporal,USD 1.100,None,Coronel Diaz,1500,None,Palermo,...,Alquiler temporario departamento amueblado de ...,https://www.argenprop.com/departamento-en-alqu...,0,0,0,0,0,0,1,0
5,2026-04-14,18713014,argenprop,alquiler_temporal,USD 1.250,$480.000,Av. Coronel Díaz,2400,None,Recoleta,...,Impecable semipiso! Balcón corrido al frente c...,https://www.argenprop.com/departamento-en-alqu...,0,1,1,0,0,1,1,0
6,2026-04-14,18712977,argenprop,alquiler_temporal,USD 1.250,$480.000,Av. Coronel Díaz,2400,None,Recoleta,...,Impecable semipiso! Balcón corrido al frente c...,https://www.argenprop.com/departamento-en-alqu...,0,1,1,0,0,1,1,0
7,2026-04-14,19280105,argenprop,alquiler_temporal,USD 1.400,$620.000,Ayacucho,2100,None,Recoleta,...,Impecable semipiso! Balcón corrido con vista a...,https://www.argenprop.com/departamento-en-alqu...,0,1,1,0,0,1,1,0
8,2026-04-14,19061700,argenprop,alquiler_temporal,USD 650,$275.000,José Andrés Pacheco De Melo,1800,None,Recoleta,...,“Torre iq callao” impecable monoambiente! Balc...,https://www.argenprop.com/departamento-en-alqu...,5,0,1,0,1,1,1,0
9,2026-04-14,19364039,argenprop,alquiler_temporal,USD 1.500,None,Guatemala,5653,None,Palermo Hollywood,...,Alquiler minimo 3 meses hasta 1 año (renovable...,https://www.argenprop.com/departamento-en-alqu...,5,0,0,0,0,0,1,0



📊 % con cada característica:
Losa_Central         13.3%
Aire_Acond           53.3%
Apto_Credito          0.0%
Cochera              10.0%
Seguridad            20.0%
Luminoso             48.3%
Balcon_Aterrazado     3.3%
dtype: object


# Venta

In [ ]:
df_venta = run_scrapper_argenprop('https://www.argenprop.com/departamentos/venta/capital-federal', 'venta', max_pages=3)


--- PÁGINA 1 ---
URL: https://www.argenprop.com/departamentos/venta/capital-federal


  → 20 propiedades encontradas
  → Jerónimo Salguero 2700 | USD 2.500.000
  → Bulnes 1600 | USD 150.000
  → Araoz 1200 | USD 330.000
  → Honduras 3900 | USD 270.000
  → Castex 3300 | USD 570.000
  → Gurruchaga 2100 | USD 98.000
  → Borges Jorge Luis 2100 | USD 240.000
  → Soler 5800 | USD 129.000
  → Jorge Luis Borges 2400 | USD 219.000
  → Av Francisco Acuña De Figueroa 1200 | USD 113.076
  → Rivadavia, Av 5200 | USD 68.000
  → Conesa 2500 | USD 105.000
  → Amenabar 2100 | USD 129.000
  → Balbin 2400 | USD 599.000
  → Jerónimo Salguero 1900 | USD 198.000
  → Santa Rosa 5000 | USD 89.000
  → Bulnes 1300 | USD 288.000
  → Fray Justo Santa María De Oro 2200 | USD 189.000
  → Avenida Gaona 1100 | USD 248.000
  → Conesa 2900 | USD 106.500

--- PÁGINA 2 ---
URL: https://www.argenprop.com/departamentos/venta/capital-federal?pagina-2
  → 20 propiedades encontradas
  → Arenales 3100 | USD 269.000
  → Armenia 2100 | USD 155.000
  → Zapata 300 | USD 91.000
  → Juan María Gutiérrez 2500 | USD 210

In [ ]:
if df_venta is not None:
    print(f"Total propiedades: {len(df_venta)}")
    display(df_venta.head(10))

    cols = ["Losa_Central","Aire_Acond","Apto_Credito",
            "Cochera","Seguridad","Luminoso","Balcon_Aterrazado"]
    print("\n📊 % con cada característica:")
    print((df_venta[cols].mean() * 100).round(1).astype(str) + "%")

Total propiedades: 60


,Fecha_Scraping,Posting_ID,Sito,Operación,Precio,Expensas,Calle,Altura,Piso,Barrio,...,Descripción,Link,Amenities,Losa_Central,Aire_Acond,Apto_Credito,Cochera,Seguridad,Luminoso,Balcon_Aterrazado
0,2026-04-10,18799423,argenprop,venta,USD 2.500.000,$2.400.000,Jerónimo Salguero,2700,None,Palermo Chico,...,“Torre bellini” de revista! Impecable piso muy...,https://www.argenprop.com/departamento-en-vent...,6,0,1,1,1,1,1,0
1,2026-04-10,15321644,argenprop,venta,USD 150.000,$260.000,Bulnes,1600,None,Botanico,...,Impecable! Balcón corrido al frente con vista ...,https://www.argenprop.com/departamento-en-vent...,0,0,1,1,1,1,1,0
2,2026-04-10,19344625,argenprop,venta,USD 330.000,$203.300,Araoz,1200,8,Palermo,...,Venta 4 ambientes con balcón palermo + cochera...,https://www.argenprop.com/departamento-en-vent...,1,0,0,0,1,1,1,0
3,2026-04-10,9070214,argenprop,venta,USD 270.000,$300.000,Honduras,3900,2,Palermo Viejo,...,"Venta semipiso 4 ambientes, dos baños completo...",https://www.argenprop.com/departamento-en-vent...,1,0,1,0,1,0,1,1
4,2026-04-10,19134132,argenprop,venta,USD 570.000,$1.000.000,Castex,3300,None,Palermo Chico,...,Palermo chico espectacular piso alto con vista...,https://www.argenprop.com/departamento-en-vent...,0,1,0,0,1,1,1,1
5,2026-04-10,19192534,argenprop,venta,USD 98.000,$150.000,Gurruchaga,2100,5,Palermo,...,Venta - departamento 1 ambiente al frente en p...,https://www.argenprop.com/departamento-en-vent...,0,1,0,0,0,0,0,0
6,2026-04-10,19106440,argenprop,venta,USD 240.000,$340.000,Borges Jorge Luis,2100,6,Palermo Soho,...,Departamento en venta 3 ambientes en palermo s...,https://www.argenprop.com/departamento-en-vent...,3,1,1,0,0,0,1,0
7,2026-04-10,19082959,argenprop,venta,USD 129.000,$240.000,Soler,5800,1,Palermo Hollywood,...,"Unidad de 2 ambientes, muy luminoso y en una e...",https://www.argenprop.com/departamento-en-vent...,0,0,0,0,0,0,1,0
8,2026-04-10,19381734,argenprop,venta,USD 219.000,$200.000,Jorge Luis Borges,2400,None,Botanico,...,Departamento dúplex de 4 ambientes reciclado a...,https://www.argenprop.com/departamento-en-vent...,0,0,1,0,0,0,1,0
9,2026-04-10,16661693,argenprop,venta,USD 113.076,None,Av Francisco Acuña De Figueroa,1200,7,Palermo,...,"Venta de departamento 1 ambiente en palermo, c...",https://www.argenprop.com/departamento-en-vent...,3,0,1,0,0,1,0,0



📊 % con cada característica:
Losa_Central         11.7%
Aire_Acond           41.7%
Apto_Credito         15.0%
Cochera              33.3%
Seguridad            31.7%
Luminoso             63.3%
Balcon_Aterrazado    10.0%
dtype: object
